#### Inisialisasi

In [7]:
import findspark
findspark.init()  # Menghubungkan VS Code ke Apache Spark lokal

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Membuat sesi Spark khusus analisis dasar dashboard
spark = SparkSession.builder \
    .appName("Analisis_Dasar_ASEAN_Dashboard_Notebook") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

print("Spark Session untuk Analisis Dasar Berhasil Aktif!")

Spark Session untuk Analisis Dasar Berhasil Aktif!


#### Load Data

In [8]:
PATH_INPUT = "hdfs://localhost:9000/Project_akhir/data_bersih_asean"
PATH_VIZ = "hdfs://localhost:9000/Project_akhir/visualisasi_asean"

print(f"Membaca data bersih dari: {PATH_INPUT}")
df = spark.read.parquet(PATH_INPUT)

# pembacaan data berhasil
total_data = df.count()
print(f"Total data bersih yang siap dianalisis: {total_data:,}")

Membaca data bersih dari: hdfs://localhost:9000/Project_akhir/data_bersih_asean
Total data bersih yang siap dianalisis: 3,857,065


#### Tujuan 1 : Persebaran Penggunaan Teknologi Radio (Pie Chart)

In [9]:
tech_overall = df.groupBy("generasi").count()

tech_overall_final = tech_overall.withColumn(
    "percentage", (F.col("count") / total_data) * 100
).orderBy(F.desc("count"))

print("Hasil Agregasi Persebaran Teknologi Radio di ASEAN:")
tech_overall_final.show()

# Simpan ke HDFS untuk visualisasi
tech_overall_final.coalesce(1).write.mode("overwrite").option("header", "true").csv(f"{PATH_VIZ}/overall_radio_percentage")
print(f"Sukses menyimpan output ke: {PATH_VIZ}/overall_radio_distribution")

Hasil Agregasi Persebaran Teknologi Radio di ASEAN:
+--------+-------+--------------------+
|generasi|  count|          percentage|
+--------+-------+--------------------+
|      3G|2285698|   59.26003321178149|
|      4G| 818725|  21.226632167204855|
|      2G| 750480|  19.457281637721945|
|      5G|   2162|0.056052983291699776|
+--------+-------+--------------------+

Sukses menyimpan output ke: hdfs://localhost:9000/Project_akhir/visualisasi_asean/overall_radio_distribution


#### Tujuan 2 : Dominasi Operator per Negara (Bar Chart Terklaster)

In [10]:
# Hitung total menara per negara dan per operator
country_counts = df.groupBy("Country", "Network").agg(F.count("*").alias("tower_count"))

# Buat window function untuk meranking operator di tiap negara
window_spec = Window.partitionBy("Country").orderBy(F.desc("tower_count"))

# Ambil peringkat top 3
top_operators = country_counts.withColumn("rank", F.row_number().over(window_spec)) \
                              .filter(F.col("rank") <= 3) \
                              .orderBy("Country", "rank")

print("Hasil Top 3 Operator Terbesar di Tiap Negara ASEAN:")
top_operators.show(30, truncate=False)

# Simpan hasil ke HDFS
top_operators.coalesce(1).write.mode("overwrite").option("header", "true").csv(f"{PATH_VIZ}/top3_operator_per_negara")
print(f"Sukses menyimpan output ke: {PATH_VIZ}/top3_operator_per_negara")

Hasil Top 3 Operator Terbesar di Tiap Negara ASEAN:
+-----------+---------------+-----------+----+
|Country    |Network        |tower_count|rank|
+-----------+---------------+-----------+----+
|Brunei     |DST            |4129       |1   |
|Brunei     |B-Mobile       |2962       |2   |
|Cambodia   |Smart          |22736      |1   |
|Cambodia   |Cellcard       |14551      |2   |
|Cambodia   |Metfone        |12061      |3   |
|East Timor |Timor Telecom  |561        |1   |
|East Timor |Telkomcel      |308        |2   |
|East Timor |Telemor        |280        |3   |
|Indonesia  |Telkomsel      |683959     |1   |
|Indonesia  |XL             |281487     |2   |
|Indonesia  |Indosat Ooredoo|210921     |3   |
|Laos       |LTC            |15151      |1   |
|Laos       |Unitel         |11846      |2   |
|Laos       |ETL Mobile     |2868       |3   |
|Malaysia   |Maxis          |194268     |1   |
|Malaysia   |Celcom         |180186     |2   |
|Malaysia   |DiGi           |138702     |3   |
|Myanmar

#### Tujuan 3 : Rasio Modernisasi vs Usia Data (Scatter Plot / Matriks)

In [11]:
# Hitung indikator teknologi modern (LTE & NR) per negara
modern_tech = df.groupBy("Country").agg(
    F.count("*").alias("total_towers"),
    F.count(F.when(F.col("radio").isin("LTE", "NR"), 1)).alias("modern_towers"),
    F.avg("data_age_days").alias("avg_data_age_days")  # Memanggil kolom data_age_days dari Preprocessing
)

# Hitung rasio persentase modernisasi
modernization_ratio = modern_tech.withColumn(
    "modernization_percentage", (F.col("modern_towers") / F.col("total_towers")) * 100
).select("Country", "total_towers", "modernization_percentage", "avg_data_age_days") \
 .orderBy(F.desc("modernization_percentage"))

print("Rasio Modernisasi Infrastruktur vs Rata-rata Usia Data per Negara:")
modernization_ratio.show(15, truncate=False)

# Simpan hasil ke HDFS
modernization_ratio.coalesce(1).write.mode("overwrite").option("header", "true").csv(f"{PATH_VIZ}/rasio_modernisasi_usia")
print(f"Sukses menyimpan output ke: {PATH_VIZ}/rasio_modernisasi_usia")

Rasio Modernisasi Infrastruktur vs Rata-rata Usia Data per Negara:
+-----------+------------+------------------------+------------------+
|Country    |total_towers|modernization_percentage|avg_data_age_days |
+-----------+------------+------------------------+------------------+
|Singapore  |133681      |45.742476492545684      |2344.7055461102286|
|Brunei     |7091        |28.740657171061912      |2591.770360945934 |
|Thailand   |1197377     |27.864824528949526      |2540.0267040236513|
|Philippines|291188      |25.354066788466557      |2391.6899373561214|
|Malaysia   |592748      |22.3978824053392        |2329.7971911331388|
|Indonesia  |1217266     |15.590183246718468      |2620.2054138141266|
|Cambodia   |49484       |14.99878748686444       |2680.170873256713 |
|Laos       |31657       |7.186404270777396       |2608.7238611461357|
|Vietnam    |301939      |5.669688248288562       |2365.675738382741 |
|Myanmar    |33485       |2.5981782887860234      |2885.9844241554965|
|East Timo

#### Tujuan 4 : Hierarki Kepemilikan Infrastruktur (Sunburst Chart)

In [12]:
# Hitung total menara global untuk persentase share
window_country = Window.partitionBy("Country")

sunburst_df = df.groupBy("Country", "Network").agg(F.count("*").alias("tower_count"))

sunburst_final = sunburst_df.withColumn("total_negara", F.sum("tower_count").over(window_country)) \
                            .withColumn("percentage_share", (F.col("tower_count") / F.col("total_negara")) * 100) \
                            .withColumn("region", F.lit("Asia Tenggara"))  # Flag statis level terdalam sunburst
                            
sunburst_final = sunburst_final.select("region", "Country", "Network", "tower_count", "percentage_share") \
                               .orderBy("Country", F.desc("tower_count"))

print("Struktur Hierarki Kepemilikan Menara Operator ASEAN:")
sunburst_final.show(15, truncate=False)

# Simpan hasil ke HDFS
sunburst_final.coalesce(1).write.mode("overwrite").option("header", "true").csv(f"{PATH_VIZ}/sunburst_asean_operator")
print(f"Sukses menyimpan output ke: {PATH_VIZ}/sunburst_asean_operator")

Struktur Hierarki Kepemilikan Menara Operator ASEAN:
+-------------+----------+---------------+-----------+-------------------+
|region       |Country   |Network        |tower_count|percentage_share   |
+-------------+----------+---------------+-----------+-------------------+
|Asia Tenggara|Brunei    |DST            |4129       |58.228740657171066 |
|Asia Tenggara|Brunei    |B-Mobile       |2962       |41.771259342828934 |
|Asia Tenggara|Cambodia  |Smart          |22736      |45.94616441678118  |
|Asia Tenggara|Cambodia  |Cellcard       |14551      |29.405464392530916 |
|Asia Tenggara|Cambodia  |Metfone        |12061      |24.373534879961202 |
|Asia Tenggara|Cambodia  |Seatel         |79         |0.15964756284859752|
|Asia Tenggara|Cambodia  |QB             |57         |0.11518874787810202|
|Asia Tenggara|East Timor|Timor Telecom  |561        |48.825065274151434 |
|Asia Tenggara|East Timor|Telkomcel      |308        |26.8059181897302   |
|Asia Tenggara|East Timor|Telemor        |280  

#### Tujuan 5 : Pertumbuhan Menara Tahunan (Line Chart)

In [13]:
# Mengelompokkan berdasarkan kolom created_year dari hasil Preprocessing
growth_df = df.groupBy("created_year").count().orderBy("created_year")

print("Tren Pertumbuhan Koleksi Menara Seluler per Tahun:")
growth_df.show()

# Simpan hasil ke HDFS
growth_df.coalesce(1).write.mode("overwrite").option("header", "true").csv(f"{PATH_VIZ}/pertumbuhan_tahunan")
print(f"Sukses menyimpan output ke: {PATH_VIZ}/pertumbuhan_tahunan")

Tren Pertumbuhan Koleksi Menara Seluler per Tahun:
+------------+-------+
|created_year|  count|
+------------+-------+
|        1980|      2|
|        2000|     10|
|        2007|      1|
|        2008|   3113|
|        2009|  28518|
|        2010|   2386|
|        2011|   5236|
|        2012|  15163|
|        2013|  68589|
|        2014| 133359|
|        2015| 526596|
|        2016|1289718|
|        2017|1366607|
|        2018|  30538|
|        2019|  89369|
|        2020|  44400|
|        2021|  34434|
|        2022|  73074|
|        2023| 126517|
|        2024|  19435|
+------------+-------+

Sukses menyimpan output ke: hdfs://localhost:9000/Project_akhir/visualisasi_asean/pertumbuhan_tahunan


#### Tujuan 6 : Top 10 Operator Terbesar di Seluruh ASEAN

In [14]:
# Hitung total menara mutlak tiap operator di tingkat ASEAN
op_total = df.groupBy("Network").agg(F.count("*").alias("total_menara_asean"))

# Cari negara tempat operator tersebut paling dominan
op_country_counts = df.groupBy("Network", "Country").agg(F.count("*").alias("cnt"))
window_op = Window.partitionBy("Network").orderBy(F.desc("cnt"))

dominant_country = op_country_counts.withColumn("rank", F.rank().over(window_op)) \
                                    .filter(F.col("rank") == 1) \
                                    .select("Network", F.col("Country").alias("Negara_Dominan"))

# Gabungkan total menara regional dengan info negara dominan
top_10_asean = op_total.join(dominant_country, on="Network", how="inner") \
                       .orderBy(F.desc("total_menara_asean")) \
                       .limit(10)

print("Top 10 Raksasa Provider Telekomunikasi Terbesar di Regional ASEAN:")
top_10_asean.show(truncate=False)

# Simpan hasil ke HDFS
top_10_asean.coalesce(1).write.mode("overwrite").option("header", "true").csv(f"{PATH_VIZ}/top10_operator_asean")
print(f"Sukses menyimpan output ke: {PATH_VIZ}/top_10_operator_asean")

Top 10 Raksasa Provider Telekomunikasi Terbesar di Regional ASEAN:
+---------------+------------------+--------------+
|Network        |total_menara_asean|Negara_Dominan|
+---------------+------------------+--------------+
|Telkomsel      |683959            |Indonesia     |
|AIS            |565878            |Thailand      |
|XL             |281487            |Indonesia     |
|Indosat Ooredoo|210921            |Indonesia     |
|Maxis          |194268            |Malaysia      |
|Celcom         |180186            |Malaysia      |
|dtac TriNet    |179395            |Thailand      |
|Globe          |172354            |Philippines   |
|TOTmobile      |155405            |Thailand      |
|dtac           |150015            |Thailand      |
+---------------+------------------+--------------+

Sukses menyimpan output ke: hdfs://localhost:9000/Project_akhir/visualisasi_asean/top_10_operator_asean
